In [ ]:
from datetime import datetime
import os
from pathlib import Path
import pandas as pd

class Field():
    EXCHANGE_PRODUCT_ID   = "exchange_product_id"
    EXCHANGE_PRODUCT_NAME = "exchange_product_name"
    DELIVERY_BASIS_NAME   = "delivery_basis_name"
    VOLUME                = "volume"
    TOTAL                 = "total"
    COUNT                 = "count"
    OIL_ID                = "oil_id"
    DELIVERY_BASIS_ID     = "delivery_basis_id"
    DELIVERY_TYPE_ID      = "delivery_type_id"
    TRADE_DATE            = "trade_date"

FIELDS = {
    1: "exchange_product_id",  # Код инструмента
    2: "exchange_product_name",  # Наименование инструмента
    3: "delivery_basis_name",  # Базис поставки
    4: "volume",  # Объём договоров в ед. измерения
    5: "total",  # Объём договоров, руб.
    14: "count",  # Количество договоров, шт.
}

NUMERIC = ["volume", "total", "count"]

def read_unit_block(path, unit="Метрическая тонна"):
    raw = pd.read_excel(path, engine="xlrd", header=None, usecols=list(FIELDS))
    raw.columns = list(FIELDS.values())

    code = raw["exchange_product_id"].astype(str)
    unit_col = code.str.extract(r"Единица измерения:\s*(.+)")[0].ffill()

    is_data = code.str.fullmatch(r"[A-Z0-9-]+")
    df = raw[is_data & unit_col.eq(unit)].reset_index(drop=True)

    for c in NUMERIC:
        df[c] = pd.to_numeric(df[c].replace("-", pd.NA), errors="coerce").astype("Int64")
    return df

def evaluate_additional_columns(df: pd.DataFrame) -> None:
    df['oil_id'] = df['exchange_product_id'].str[:4]
    df['delivery_basis_id'] = df['exchange_product_id'].str[4:7]
    df['delivery_type_id'] = df['exchange_product_id'].str[-1]


# SCRIPT_DIR = Path("/root/projects/learning/DataBase/Practice/second")
# DOWNLOAD_DIR = SCRIPT_DIR / "download"

# for file in DOWNLOAD_DIR.iterdir():
#     print(file.name)
file_path = "/root/projects/learning/DataBase/Practice/second/download/01.02.2023_68e09872-089c-4e37-948e-be5f7819b0e8.xls"

# result_dfs: list[tuple[datetime, pd.DataFrame]] = []

df = read_unit_block(file_path)
filtered_df = df[df["count"] > 0]

evaluate_additional_columns(filtered_df)
# filtered_df['oil_id'] = filtered_df['exchange_product_id'].str[:4]
# filtered_df['delivery_basis_id'] = filtered_df['exchange_product_id'].str[4:7]
# filtered_df['delivery_type_id'] = filtered_df['exchange_product_id'].str[-1]
filtered_df

In [4]:
from datetime import datetime
import os
from pathlib import Path
import pandas as pd

class Field:
    EXCHANGE_PRODUCT_ID = "exchange_product_id"
    EXCHANGE_PRODUCT_NAME = "exchange_product_name"
    DELIVERY_BASIS_NAME = "delivery_basis_name"
    VOLUME = "volume"
    TOTAL = "total"
    COUNT = "count"
    OIL_ID = "oil_id"
    DELIVERY_BASIS_ID = "delivery_basis_id"
    DELIVERY_TYPE_ID = "delivery_type_id"
    TRADE_DATE = "trade_date"


COLUMNS = {
    1: Field.EXCHANGE_PRODUCT_ID,  # Код инструмента
    2: Field.EXCHANGE_PRODUCT_NAME,  # Наименование инструмента
    3: Field.DELIVERY_BASIS_NAME,  # Базис поставки
    4: Field.VOLUME,  # Объём договоров в ед. измерения
    5: Field.TOTAL,  # Объём договоров, руб.
    14: Field.COUNT,  # Количество договоров, шт.
}


def read_xls(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, engine="xlrd", header=None, usecols=list(COLUMNS))
    df.columns = list(COLUMNS.values())
    return df


def extract_date(df: pd.DataFrame) -> datetime:
    marker = df[Field.EXCHANGE_PRODUCT_ID].astype(str)
    trade_date = marker.str.extract(r"Дата торгов:\s*(\d{2}\.\d{2}\.\d{4})")[0].dropna()
    trade_date = (
        pd.to_datetime(trade_date.iloc[0], format="%d.%m.%Y").date()
        if len(trade_date)
        else None
    )
    if not trade_date:
        raise Exception("Не удалось получить дату из документа")

    return trade_date


def filter_df(df: pd.DataFrame, unit="Метрическая тонна"):
    marker = df[Field.EXCHANGE_PRODUCT_ID].astype(str)
    unit_col = marker.str.extract(r"Единица измерения:\s*(.+)")[0].ffill()

    is_data = marker.str.fullmatch(r"[A-Z0-9-]+")
    df = df[is_data & unit_col.eq(unit)].reset_index(drop=True)

    for c in [Field.VOLUME, Field.TOTAL, Field.COUNT]:
        df[c] = pd.to_numeric(df[c].replace("-", pd.NA), errors="coerce").astype(
            "Int64"
        )

    df = df[df["count"] > 0]

    return df


def evaluate_additional_columns(df: pd.DataFrame, trade_date: datetime) -> None:
    df[Field.OIL_ID] = df[Field.EXCHANGE_PRODUCT_ID].str[:4]
    df[Field.DELIVERY_BASIS_ID] = df[Field.EXCHANGE_PRODUCT_ID].str[4:7]
    df[Field.DELIVERY_TYPE_ID] = df[Field.EXCHANGE_PRODUCT_ID].str[-1]
    df[Field.TRADE_DATE] = trade_date


def collect_data_from_xls(DOWNLOAD_DIR: Path) -> pd.DataFrame:
    result_data_dfs: list[pd.DataFrame] = []
    for file in DOWNLOAD_DIR.iterdir():
        df = read_xls(str(file))
        trade_date = extract_date(df)
        filtered_df = filter_df(df)
        evaluate_additional_columns(filtered_df, trade_date)
        result_data_dfs.append(filtered_df)

    result_df = pd.concat(result_data_dfs, axis=0, ignore_index=True)
    return result_df

SCRIPT_DIR = Path("/root/projects/learning/DataBase/Practice/second")
DOWNLOAD_DIR = SCRIPT_DIR / "download"
DOWNLOAD_DIR.mkdir(exist_ok=True)

d = collect_data_from_xls(DOWNLOAD_DIR)
d

,exchange_product_id,exchange_product_name,delivery_basis_name,volume,total,count,oil_id,delivery_basis_id,delivery_type_id,trade_date
0,A100UFM060F,"Бензин (АИ-100-К5), Уфа-группа станций (ст. от...",Уфа-группа станций,120,6905220,2,A100,UFM,F,2023-03-07
1,A592ACH005A,"Бензин (АИ-92-К5) по ГОСТ, Ачинский НПЗ (самов...",Ачинский НПЗ,25,1342500,1,A592,ACH,A,2023-03-07
2,A592AKR060F,"Бензин (АИ-92-К5) по ГОСТ, ст. Аксарайская II ...",ст. Аксарайская II,1080,54463920,15,A592,AKR,F,2023-03-07
3,A592ALL060F,"Бензин (АИ-92-К5) по ГОСТ, ст. Аллагуват (ст. ...",ст. Аллагуват,780,37441980,11,A592,ALL,F,2023-03-07
4,A592ANK060F,"Бензин (АИ-92-К5) по ГОСТ, Ангарск-группа стан...",Ангарск-группа станций,540,29139300,9,A592,ANK,F,2023-03-07
...,...,...,...,...,...,...,...,...,...,...
150688,TKM7KOB065J,"Мазут топочный TKM-16, не облагаемый акцизом, ...",ст. Комбинатская,130,2931500,2,TKM7,KOB,J,2025-10-17
150689,TPBUUPG025A,"Топливо печное бытовое, Сургутское УПГ (самовы...",Сургутское УПГ,50,4265300,2,TPBU,UPG,A,2025-10-17
150690,TRD-BTT060R,Топливо для реактивных двигателей марок РТ в/с...,Батайск БП,960,78164160,1,TRD-,BTT,R,2025-10-17
150691,TRD-MHA060R,Топливо для реактивных двигателей марок РТ в/с...,МАУ БП,1080,85986000,4,TRD-,MHA,R,2025-10-17
